# 02 — Silver to Gold
Reads the cleaned Silver tables, builds one dimension (`dim_employee`) and two facts
(`fact_attendance`, `fact_performance`), checks for orphan keys, writes to Gold,
and registers the tables in the catalog.

In [0]:
STORAGE_ACCOUNT = "terrsthranalyticdevov4j3"

df_employees_clean = spark.read.format("delta").load(f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/employees/")
df_attendance_clean = spark.read.format("delta").load(f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/attendance/")
df_performance = spark.read.format("delta").load(f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/performance/")

In [0]:
print("Employees:", df_employees_clean.count())
print("Attendance:", df_attendance_clean.count())
print("Performance:", df_performance.count())

## dim_employee

In [0]:
df_dim_employee = df_employees_clean.select(
    "EmployeeID", "Name", "Department", "Position", "HireDate", "Salary"
)

df_dim_employee.show(25, truncate=False)

## fact_attendance
Keeps `Year` + `Month` + `AttendanceMonth` (a real date) so Power BI can build the Year → Quarter → Month drill-down hierarchy directly off this fact table.

In [0]:
from pyspark.sql.functions import col

df_fact_attendance = df_attendance_clean.select(
    "EmployeeID",
    "Year",
    "MonthNum",
    "AttendanceMonth",
    "DaysPresent",
    "DaysAbsent",
    "LateArrivals"
).withColumnRenamed("MonthNum", "Month")

df_fact_attendance = df_fact_attendance \
    .withColumn("DaysPresent", col("DaysPresent").cast("int")) \
    .withColumn("DaysAbsent", col("DaysAbsent").cast("int")) \
    .withColumn("LateArrivals", col("LateArrivals").cast("int"))

df_fact_attendance.show(25, truncate=False)

## fact_performance
Adds `ReviewYear` / `ReviewQuarter` / `ReviewMonth` so Performance can be filtered or joined against Attendance at month grain.

In [0]:
from pyspark.sql.functions import year, quarter, month

df_fact_performance = df_performance.select(
    "EmployeeID",
    "ReviewDate",
    "PerformanceScore",
    "ManagerComment"
).withColumn("ReviewYear", year("ReviewDate")) \
 .withColumn("ReviewQuarter", quarter("ReviewDate")) \
 .withColumn("ReviewMonth", month("ReviewDate"))

df_fact_performance.show(25, truncate=False)

## Orphan key check
Any EmployeeID in a fact table that doesn't exist in `dim_employee` will break Power BI relationships silently. Check before writing, not after.

In [0]:
dim_ids = set(row.EmployeeID for row in df_dim_employee.select("EmployeeID").distinct().collect())

attendance_ids = set(row.EmployeeID for row in df_fact_attendance.select("EmployeeID").distinct().collect())
performance_ids = set(row.EmployeeID for row in df_fact_performance.select("EmployeeID").distinct().collect())

print("Attendance IDs not in dim_employee:", attendance_ids - dim_ids)
print("Performance IDs not in dim_employee:", performance_ids - dim_ids)

## Write to Gold

In [0]:
df_dim_employee.write.mode("overwrite").option("overwriteSchema", "true").format("delta") \
    .save(f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/dim_employee/")

df_fact_attendance.write.mode("overwrite").option("overwriteSchema", "true").format("delta") \
    .save(f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/fact_attendance/")

df_fact_performance.write.mode("overwrite").option("overwriteSchema", "true").format("delta") \
    .save(f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/fact_performance/")

print("Gold layer written.")

## Register in catalog
Assumes `terr_dbx_hranalytics_dev_ov4j3_7405611111933300.default` already exists. If it doesn't, you'll hit the same metastore-storage-root error we fixed for `dev` earlier — create it once via Catalog Explorer with an explicit storage location.

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS `terr_dbx_hranalytics_dev_ov4j3_7405611111933300`.default.dim_employee
USING DELTA
LOCATION 'abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/dim_employee/'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `terr_dbx_hranalytics_dev_ov4j3_7405611111933300`.default.fact_attendance
USING DELTA
LOCATION 'abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/fact_attendance/'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `terr_dbx_hranalytics_dev_ov4j3_7405611111933300`.default.fact_performance
USING DELTA
LOCATION 'abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/fact_performance/'
""")

In [0]:
spark.sql("SHOW TABLES IN `terr_dbx_hranalytics_dev_ov4j3_7405611111933300`.default").show()

## Duplicate check on the registered table

In [0]:
%sql
SELECT
    EmployeeID,
    COUNT(*) AS count
FROM `terr_dbx_hranalytics_dev_ov4j3_7405611111933300`.default.dim_employee
GROUP BY EmployeeID
HAVING COUNT(*) > 1
ORDER BY count DESC;

## Sanity check: attendance rate per employee
Same formula you'd use as a Power BI measure — run here first so you know what the "right answer" looks like before building the report.

In [0]:
df_attendance_gold = spark.read.format("delta").load(f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/fact_attendance/")

from pyspark.sql import functions as F

df_rate = df_attendance_gold.groupBy("EmployeeID").agg(
    F.sum("DaysPresent").alias("TotalPresent"),
    F.sum("DaysAbsent").alias("TotalAbsent")
).withColumn(
    "AttendanceRate",
    F.col("TotalPresent") / (F.col("TotalPresent") + F.col("TotalAbsent"))
)

df_rate.orderBy("AttendanceRate").show(25)

df_rate.select(
    F.min("AttendanceRate").alias("Min"),
    F.max("AttendanceRate").alias("Max"),
    F.avg("AttendanceRate").alias("Avg"),
    F.stddev("AttendanceRate").alias("StdDev")
).show()

In [0]:
%sql
-- MSCK REPAIR TABLE `terr_dbx_hranalytics_dev_ov4j3_7405611111933300`.default.fact_attendance SYNC METADATA;
-- REFRESH TABLE `terr_dbx_hranalytics_dev_ov4j3_7405611111933300`.default.fact_attendance;
MSCK REPAIR TABLE `terr_dbx_hranalytics_dev_ov4j3_7405611111933300`.default.fact_performance SYNC METADATA;
REFRESH TABLE `terr_dbx_hranalytics_dev_ov4j3_7405611111933300`.default.fact_performance;

In [0]:
%sql
DESCRIBE TABLE `terr_dbx_hranalytics_dev_ov4j3_7405611111933300`.default.fact_attendance;

In [0]:
%sql
DESCRIBE TABLE terr_dbx_hranalytics_dev_ov4j3_7405611111933300.default.dim_employee;